# Exploring sample trip logs & testing PySpark logic

In [0]:
import requests
import zipfile
import io
import os
from pyspark.sql import SparkSession
import glob
import xml.etree.ElementTree as ET

## Connecting to data lake

In [0]:
storage_name = os.environ["STORAGE_ACCOUNT_NAME"]
storage_key = os.environ["STORAGE_ACCOUNT_KEY"]

spark.conf.set(
    f"fs.azure.account.key.{storage_name}.dfs.core.windows.net", storage_key
)

# Define your true Cloud Bronze zone
cloud_bronze_historical = (
    f"abfss://bronze@{storage_name}.dfs.core.windows.net/historical_tripdata/"
)

## dynamic zip files url extraction script

In [0]:
def get_filtered_citibike_zip_urls(start_year: int = 2020, end_year: int = None):
    """Fetches Citi Bike tripdata zip URLs from S3, filtered by year range and excluding Jersey City."""
    base_url = "https://s3.amazonaws.com/tripdata"
    zip_urls = []

    ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
    marker = None

    print(
        f"Scanning S3 Bucket for non-JC files from {start_year}"
        + (f" to {end_year}..." if end_year else " onward...")
    )

    while True:
        params = {"marker": marker} if marker else {}
        response = requests.get(base_url, params=params)
        response.raise_for_status()

        root = ET.fromstring(response.content)

        for content in root.findall("s3:Contents", ns):
            key = content.find("s3:Key", ns).text

            # 1. Must be a zip file AND exclude Jersey City 'JC-' files
            if key and key.endswith(".zip") and not key.startswith("JC"):

                # 2. The first 4 characters of the key always represent the year (e.g. '202401...' -> 2024)
                if key[:4].isdigit():
                    file_year = int(key[:4])

                    # 3. Apply your temporal bounding box
                    if file_year >= start_year:
                        if end_year and file_year > end_year:
                            continue  # Skip if it sits past your ceiling

                        zip_urls.append(f"{base_url}/{key}")

        is_truncated = root.find("s3:IsTruncated", ns)
        if is_truncated is not None and is_truncated.text.lower() == "true":
            marker = root.findall("s3:Contents", ns)[-1].find("s3:Key", ns).text
        else:
            break

    print(f"Discovered {len(zip_urls)} targeted ZIP files.")
    return zip_urls

## Extracting more recent datasets

In [0]:
target_urls = get_filtered_citibike_zip_urls(start_year=2022)
print(target_urls)

### historical data ingestion script flattens folder embeddings

In [0]:
def get_existing_adls_files(adls_path):
    """Scans the Bronze Data Lake and returns a set of filenames already safely inside."""
    try:
        return {
            f.name
            for f in dbutils.fs.ls(adls_path)
            if f.name.endswith(".csv")
        }
    except Exception:
        return set()  # Return empty set if the bronze folder hasn't been created yet

In [0]:
def extract_and_land_in_adls(zip_urls, target_adls_path):
    staging_dir = "/tmp/citibike_scratchpad/"
    os.makedirs(staging_dir, exist_ok=True)

    # 1. Grab our 'Already Done' checklist from Azure
    landed_files = get_existing_adls_files(target_adls_path)
    print(
        f"Index Check: Found {len(landed_files)} CSVs already safe in ADLS. Will skip these."
    )

    for url in zip_urls:
        print(f"\nTargeting URL: {url}")

        # Step A: Stream download to Driver Disk in 1MB chunks (Uses 0% RAM)
        root_zip_path = os.path.join(staging_dir, "master_stream.zip")
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(root_zip_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    f.write(chunk)

        # Step B: Create a dynamic processing Queue starting with our root zip
        inspection_queue = [root_zip_path]

        while inspection_queue:
            current_zip_path = inspection_queue.pop(0)

            try:
                with zipfile.ZipFile(current_zip_path, "r") as z:
                    for internal_path in z.namelist():
                        base_name = os.path.basename(internal_path)

                        # Skip directories, Mac resource forks, and Jersey City
                        if (
                            not base_name
                            or "__MACOSX" in internal_path
                            or base_name.startswith("JC-")
                            or base_name.startswith("._")
                        ):
                            continue

                        # ==========================================
                        # SCENARIO 1: We hit a raw CSV
                        # ==========================================
                        if base_name.endswith(".csv"):
                            if base_name in landed_files:
                                print(
                                    f"  -> [SKIPPED] '{base_name}' (Already landed)"
                                )
                                continue

                            local_csv = os.path.join(staging_dir, base_name)
                            final_adls = os.path.join(
                                target_adls_path, base_name
                            )

                            print(f"  -> Extracting CSV: {base_name}...")
                            with z.open(internal_path) as z_source, open(
                                local_csv, "wb"
                            ) as local_target:
                                local_target.write(z_source.read())

                            print(f"  -> Moving to Azure Data Lake...")
                            dbutils.fs.mv(
                                f"file:{local_csv}", final_adls, recurse=False
                            )
                            landed_files.add(
                                base_name
                            )  # Add to checklist instantly

                        # ==========================================
                        # SCENARIO 2: We hit a nested .zip or .csv.zip
                        # ==========================================
                        elif base_name.endswith(".zip"):
                            print(
                                f"  -> [Nested Archive Discovered]: {base_name}"
                            )

                            nested_target_path = os.path.join(
                                staging_dir, f"sub_{base_name}"
                            )

                            # Extract the nested zip to our scratch disk
                            with z.open(internal_path) as z_source, open(
                                nested_target_path, "wb"
                            ) as local_target:
                                local_target.write(z_source.read())

                            # Throw it to the back of the line to be cracked open next!
                            inspection_queue.append(nested_target_path)

            except zipfile.BadZipFile:
                print(
                    f"  [!] WARNING: Corrupt archive layer skipped -> {current_zip_path}"
                )

            finally:
                # Delete the specific sub-zip we just finished to keep /tmp totally empty
                if (
                    os.path.exists(current_zip_path)
                    and current_zip_path != root_zip_path
                ):
                    os.remove(current_zip_path)

    # Clean up the master stream file when the URL is fully exhausted
    if os.path.exists(root_zip_path):
        os.remove(root_zip_path)

    print("\nIngestion pass complete.")

# Ingestiong the historical trip data and loading onto the datalake

In [0]:
extract_and_land_in_adls(target_urls, cloud_bronze_historical)

# Profiling All schemas in the landed data

In [0]:
from collections import defaultdict


def profile_adls_bronze_schemas(adls_folder_path):
    """Scans an ADLS Gen2 directory, inspects the CSV headers via PySpark,

    and groups files by their exact column signature.
    """
    print(f"Querying storage layer: {adls_folder_path}...")

    try:
        # Fetch all FileInfo objects in the directory
        raw_files = dbutils.fs.ls(adls_folder_path)
        csv_files = [f for f in raw_files if f.name.endswith(".csv")]
    except Exception as e:
        print(f"[!] Storage Error. Double check your path. Details: {e}")
        return {}

    total_files = len(csv_files)
    print(f"Discovered {total_files} CSVs. Spawning lightweight Spark jobs...")

    schema_versions = defaultdict(list)

    for i, f in enumerate(csv_files, 1):
        file_path = f.path
        file_name = f.name

        # .limit(1) acts as a hard circuit breaker to prevent full file scans
        cols = tuple(
            spark.read.option("header", "true")
            .csv(file_path)
            .limit(1)
            .columns
        )

        schema_versions[cols].append(file_name)

        if i % 10 == 0 or i == total_files:
            print(f"  -> Profiled {i}/{total_files} files...")

    # Sort schemas by the number of files using them (Largest group first)
    ranked_schemas = sorted(
        schema_versions.items(), key=lambda item: len(item[1]), reverse=True
    )

    print(
        f"\n"
        + "=" * 60
        + f"\n ANALYSIS COMPLETE: Found {len(ranked_schemas)} distinct schema variations\n"
        + "=" * 60
    )

    structured_report = {}

    for index, (schema_tuple, file_list) in enumerate(ranked_schemas, 1):
        print(f"=== Schema Version {index} ===")
        print(f"Usage        : {len(file_list)} files ({round((len(file_list)/total_files)*100, 1)}% of total)")
        print(f"Total Columns: {len(schema_tuple)}")
        print(f"Columns      : {list(schema_tuple)}")
        print(f"Sample files : {file_list[:2]}\n")

        structured_report[f"v{index}"] = {
            "column_count": len(schema_tuple),
            "columns": list(schema_tuple),
            "file_count": len(file_list),
            "files": file_list,
        }

    return structured_report


In [0]:
# Run it using the variable we set in the last notebook block:
schema_report = profile_adls_bronze_schemas(cloud_bronze_historical)

In [0]:
df_trips.count()

In [0]:
df_trips.limit(20).toPandas()

## Older data sets follow different folder embeddings and schemas

In [0]:
extract_csvs_from_zip_urls([zip_urls[0]])

In [0]:
# Read all extracted CSVs beginning with 2014 in the historical folder
# PySpark will automatically merge split CSVs from the same month
df_trips_2014 = spark.read.option("header", "true") \
    .csv(f"{bronze_historical_path}/2014*.csv")

# Profile the schema to map out data types
print("The Older Schema:")
df_trips_2014.printSchema()

### script to detect all different schemas present in the csv files

In [0]:
# Get all CSV files in the bronze directory
csv_files = glob.glob(f"{bronze_historical_path}/*.csv")

# Dictionary to group files by their exact column structure
schema_versions = {}

for file in csv_files:
    # PySpark reads ONLY the first row to get the header names
    # No full data scan is triggered here
    columns = tuple(spark.read.option("header", "true").csv(file).columns)

    file_name = os.path.basename(file)

    # Group the files by their schema tuple
    if columns not in schema_versions:
        schema_versions[columns] = []

    schema_versions[columns].append(file_name)

# Output the findings
print(f"Found {len(schema_versions)} different schema structures across the files:\n")

for i, (schema, files) in enumerate(schema_versions.items(), 1):
    print(f"=== Schema Version {i} ===")
    print(f"Total Columns: {len(schema)}")
    print(f"Columns: {list(schema)}")
    print(f"Files using this schema: {len(files)}")
    print(f"Sample Files: {files[:2]}\n")